# 04 — Savunma Değerlendirmesi (Defense Evaluation)

**Cold-Start Sosyal-Grafik Sızıntısı — Simülasyon Çalışması**

Makalenin VI. bölümündeki savunmaları **spekülasyondan ölçüme** taşır. Her savunmayı bir düğmeye bağlayıp, aynı yeniden-sıralama katmanını hem **saldırgana** (gizlilik) hem **meşru kullanıcıya** (fayda) uygulayarak **gizlilik-sızıntısı ↔ öneri-faydası (Pareto)** eğrilerini çıkarır.

Değerlendirilen savunmalar:
- **Gürültü** — skorlara DP/MemGuard tarzı çıktı pertürbasyonu (knob = σ).
- **Geniş havuz** — sıcaklık (temperature) ile daha geniş, düşük-hassasiyetli aday havuzundan örnekleme (knob = τ).
- **Zorunlu çeşitlendirme** — MMR ile alaka↔çeşitlilik dengesi (knob = λ; 1 = saf alaka).
- **Gözlem bütçesi / rate-limiting** — tekrarlı sorgularla birikimli saldırı; gürültünün tek başına neden yetmediğini gösterir.

> Bu bölüm, makalede şu an yalnızca "öneri" olan savunmaları **ölçülmüş katkıya** çevirir — muhtemelen en yüksek puanı alacağınız kısım.


## 0. Kurulum ve Drive bağlama

In [ ]:
!pip install -q networkx numpy pandas scipy scikit-learn matplotlib

In [ ]:
import os, json, pickle
import numpy as np, pandas as pd, networkx as nx, torch
import matplotlib.pyplot as plt

def _in_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False

if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT = os.path.abspath('./cold_start_leakage')

DATA_DIR = os.path.join(ROOT, 'data'); MODELS_DIR = os.path.join(ROOT, 'models')
np.random.seed(0)
print("Veri  :", DATA_DIR); print("Model :", MODELS_DIR)

## 1. Şema, motor (skor + savunma katmanı), metrikler — kendi kendine yeterli

In [ ]:
class SocialGraphDataset:
    def __init__(s, name, G, private, community, node_list, features, meta=None):
        s.name, s.G, s.private, s.community = name, G, private, community
        s.node_list, s.features, s.meta = node_list, features, (meta or {})
    def followers(s, t): return set(s.G.predecessors(t))
    def following(s, t): return set(s.G.successors(t))
    def neighborhood(s, t): return s.followers(t) | s.following(t)

def load_dataset(name, root=DATA_DIR):
    d = os.path.join(root, name)
    G = pickle.load(open(os.path.join(d, "graph.pkl"), "rb"))
    ndf = pd.read_csv(os.path.join(d, "nodes.csv")); X = np.load(os.path.join(d, "features.npy"))
    return SocialGraphDataset(name, G, dict(zip(ndf.node_id, ndf["private"])),
                              dict(zip(ndf.node_id, ndf.community)), list(ndf.node_id), X)

class Engine:
    # taban skor + savunma yeniden-sıralama katmanı (saldırgan ve faydaya AYNI uygulanır)
    def __init__(self, E, k_gate=5):
        self.E = torch.nn.functional.normalize(torch.as_tensor(E, dtype=torch.float32), dim=1)
        self.n = self.E.shape[0]; self.k_gate = k_gate
    def query_context(self, target):  return self.E[target]                 # cold saldırgan
    def query_observer(self, contacts):                                     # warm meşru kullanıcı
        return torch.nn.functional.normalize(self.E[list(contacts)].mean(0, keepdim=True), dim=1)[0]
    def base_scores(self, q): return (self.E @ q).clone()

    def defended_topk(self, scores, k=20, defense="none", knob=None, exclude=None, rng=None):
        rng = rng or np.random.default_rng(0)
        s = scores.clone()
        for x in (exclude or []): s[x] = -1e9
        if defense == "none":
            return torch.topk(s, k).indices.tolist()
        if defense == "noise":
            s = s + torch.tensor(rng.normal(scale=knob, size=s.shape[0]), dtype=torch.float32)
            return torch.topk(s, k).indices.tolist()
        if defense == "pool":                       # sıcaklık ile geniş havuz örnekleme
            M = min(self.n - len(exclude or []), 200)
            cand = torch.topk(s, M).indices.tolist()
            logits = np.array([float(s[c]) for c in cand]) / max(knob, 1e-6)
            logits -= logits.max(); p = np.exp(logits); p /= p.sum()
            return [cand[i] for i in rng.choice(len(cand), size=k, replace=False, p=p)]
        if defense == "diversify":                  # MMR, lam=knob
            M = min(self.n - len(exclude or []), 100)
            cand = torch.topk(s, M).indices.tolist()
            rel = {c: float(s[c]) for c in cand}; Ec = {c: self.E[c] for c in cand}
            selected = []
            while len(selected) < k and cand:
                best, bv = None, -1e9
                for c in cand:
                    div = max((float(Ec[c] @ Ec[d]) for d in selected), default=0.0)
                    val = knob * rel[c] - (1 - knob) * div
                    if val > bv: bv, best = val, c
                selected.append(best); cand.remove(best)
            return selected
        raise ValueError(defense)

def ndcg_at_k(ranked, relevant, k):
    dcg = sum(1.0/np.log2(i+2) for i, c in enumerate(ranked[:k]) if c in relevant)
    ideal = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), k)))
    return dcg/ideal if ideal > 0 else 0.0

def privacy_leakage(eng, ds, targets, defense, knob, k=20, rng=None):
    rng = rng or np.random.default_rng(1); ps = []
    for t in targets:
        sc = eng.base_scores(eng.query_context(t))
        tk = eng.defended_topk(sc, k=k, defense=defense, knob=knob, exclude=[t], rng=rng)
        ps.append(len(set(tk) & ds.neighborhood(t))/k)
    return float(np.mean(ps))

def utility(eng, ds, warm_users, holdout, defense, knob, k=20, rng=None):
    rng = rng or np.random.default_rng(2); rec, ndcg = [], []
    for u in warm_users:
        held = holdout[u]; retained = ds.neighborhood(u) - held
        if not held or not retained: continue
        sc = eng.base_scores(eng.query_observer(retained))
        tk = eng.defended_topk(sc, k=k, defense=defense, knob=knob,
                               exclude=list(retained)+[u], rng=rng)
        rec.append(len(set(tk) & held)/len(held)); ndcg.append(ndcg_at_k(tk, held, k))
    return float(np.mean(rec)), float(np.mean(ndcg))

## 2. Kurulum: hedefler, meşru kullanıcılar ve held-out kenarlar

In [ ]:
DATASET = "sbm_default"      # Colab'da "lfr_default" ile de çalıştırılabilir
K = 20
rng = np.random.default_rng(7)

ds = load_dataset(DATASET)
E = np.load(os.path.join(MODELS_DIR, f"{DATASET}__lightgcn__emb.npy"))
eng = Engine(E, k_gate=5)

targets = [t for t in ds.node_list if ds.private[t] and ds.G.degree(t) >= 4]
warm_users = [u for u in ds.node_list if not ds.private[u] and ds.G.degree(u) >= 8][:120]
holdout = {}
for u in warm_users:
    N = list(ds.neighborhood(u))
    holdout[u] = set(rng.choice(N, size=max(1, int(0.3*len(N))), replace=False))

base_leak = privacy_leakage(eng, ds, targets, "none", None, rng=rng)
base_rec, base_ndcg = utility(eng, ds, warm_users, holdout, "none", None, rng=rng)
print(f"{DATASET}: private hedef={len(targets)}, meşru kullanıcı={len(warm_users)}")
print(f"SAVUNMASIZ taban -> sızıntı(prec@20)={base_leak:.3f} | fayda recall@20={base_rec:.3f}, ndcg={base_ndcg:.3f}")

## 3. Savunma taramaları → gizlilik-fayda Pareto noktaları

Her savunma kendi düğmesi boyunca süpürülür; her ayar için (fayda, sızıntı) çifti ölçülür. *Not: held-out kenarlar eğitimde de görülmüştü; mutlak fayda değil, savunmalar arası **göreli** takas anlamlıdır.*


In [ ]:
sweeps = {
    "noise":     [0.0, 0.02, 0.05, 0.1, 0.2, 0.4],
    "pool":      [1e-3, 0.02, 0.05, 0.1, 0.2, 0.5],
    "diversify": [1.0, 0.9, 0.7, 0.5, 0.3, 0.1],
}
records = []
for d, knobs in sweeps.items():
    for kb in knobs:
        leak = privacy_leakage(eng, ds, targets, d, kb, k=K, rng=rng)
        rec, nd = utility(eng, ds, warm_users, holdout, d, kb, k=K, rng=rng)
        records.append({"defense": d, "knob": kb, "leakage_prec@20": round(leak,3),
                        "utility_recall@20": round(rec,3), "utility_ndcg@20": round(nd,3)})
res = pd.DataFrame(records)
res

## 4. Pareto eğrileri — hangi savunma faydayı koruyup sızıntıyı düşürüyor?

In [ ]:
plt.figure(figsize=(7.5,5.5))
colors = {"noise":"#C0504D", "pool":"#4F81BD", "diversify":"#2E9E8F"}
for d in sweeps:
    sub = res[res.defense == d]
    plt.plot(sub["utility_recall@20"], sub["leakage_prec@20"], "o-", color=colors[d], label=d)
plt.scatter([base_rec], [base_leak], marker="*", s=260, color="black", zorder=5, label="savunmasız")
plt.xlabel("öneri faydası  (recall@20)  →  yüksek iyi")
plt.ylabel("gizlilik sızıntısı  (precision@20)  ↓  düşük iyi")
plt.title(f"Gizlilik–Fayda Pareto takası ({DATASET})\nideal bölge: sağ-alt")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Sağ-alta en yakın eğri, verili fayda için en düşük sızıntıyı sunan savunmadır.")

## 5. Dürüst nüans: naif çeşitlendirme gizlilik çözümü değildir

MMR tablosuna dikkat: **hafif** çeşitlendirme (λ≈0.5–0.7) sızıntıyı düşürmez, hatta biraz artırabilir; yalnız **agresif** çeşitlendirme (λ≤0.3) sızıntıyı düşürür, o da faydayı bozarak. Yani çeşitlendirmeyi "ekleyince gizlilik gelir" sanmak yanlış — bu, makalenizin *gizlilik tasarlanmış bir hedef olmalı* tezini güçlendiren ölçülmüş bir bulgudur.


In [ ]:
res[res.defense == "diversify"][["knob","leakage_prec@20","utility_recall@20","utility_ndcg@20"]]

## 6. Gözlem bütçesi / rate-limiting — gürültü neden tek başına yetmez?

Sabırlı bir saldırgan, aynı hedefe **Q** kez sorgu atıp sonuçları biriktirerek (oy sayımı) gürültüyü ortalamayla yok edebilir. Aşağıda gürültü açıkken (σ=0.2) Q arttıkça sızıntının savunmasız seviyeye geri tırmandığını görüyoruz → bu yüzden **gözlem bütçesi/rate-limiting** gereklidir. Makalenizin "birikimli düşük-yoğunluklu sinyaller" tezinin doğrudan kanıtı.


In [ ]:
NOISE = 0.2; Q_GRID = [1, 2, 5, 10, 20, 40]
agg_leak = []
for Q in Q_GRID:
    ps = []
    for t in targets[:80]:
        sc = eng.base_scores(eng.query_context(t)); counts = np.zeros(eng.n)
        for _ in range(Q):
            tk = eng.defended_topk(sc, k=K, defense="noise", knob=NOISE, exclude=[t], rng=rng)
            for c in tk: counts[c] += 1
        agg_top = list(np.argsort(-counts)[:K])
        ps.append(len(set(agg_top) & ds.neighborhood(t))/K)
    agg_leak.append(np.mean(ps))

plt.figure(figsize=(7,4.5))
plt.plot(Q_GRID, agg_leak, "o-", color="#C0504D", label="gürültü açık (σ=0.2), Q sorgu biriktiriliyor")
plt.axhline(base_leak, ls="--", color="black", label=f"savunmasız taban ({base_leak:.2f})")
plt.xlabel("saldırganın sorgu sayısı  Q"); plt.ylabel("birikimli sızıntı (prec@20)")
plt.title("Gürültü tek başına yetmez: tekrarlı gözlemle sinyal geri kazanılıyor")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("Sonuç: çıktı gürültüsü Q ile aşınır -> gözlem bütçesi/rate-limiting zorunlu tamamlayıcı savunmadır.")

## Özet — bu notebook ne kuruyor?

- Makale VI'daki savunmalar artık **spekülatif liste değil, ölçülmüş takas eğrileri.**
- Her savunma için **gizlilik–fayda Pareto** noktaları; hangisinin verili fayda için en az sızıntı sunduğu görülebiliyor.
- **Dürüst nüans:** naif çeşitlendirme gizliliği iyileştirmez; yalnız agresif çeşitlendirme (fayda bedeliyle) işe yarar.
- **Gözlem bütçesi:** gürültü tek başına yetmez — tekrarlı sorguyla aşınır, dolayısıyla rate-limiting gereklidir.

Bu dört notebook (01 veri → 02 model → 03 saldırgan/metrik → 04 savunma) birlikte, etik/ihlal sorunu olmayan, ground-truth'lu, istatistiksel ve savunma-değerlendirmeli tam bir simülasyon çalışması oluşturuyor. PoPETs / WPES / ACM TOPS için savunulabilir bir paket.
